# So you want to run a Stroop experiment?
Our group project gives you all the tools you need to plan and execute your study.

With this notebook, you can simulate data, compare frequentist and Bayesian study designs, run a Stroop experiment, and analyse the results. We describe all functionality in the sections below.

## 1. Setup

All source code is loaded from `src/py`, while this notebook exposes the high-level interfaces we need.

In [ ]:
from src.py.experiment import load_experiment_data, run_stroop
from src.py.simulate import HEDGES_G, simulate_stroop
from src.py.analysis import (
    stroop_bayes_design, stroop_bayes_t_test, stroop_experiment_plot,
    stroop_experiment_t_test, stroop_histplot, stroop_power, stroop_t_test,
)
from src.py.power_analysis_app import app

## 2. Plan your study

We can plan our study by simulating data and estimating the required sample size. We provide both frequentist power and a Bayesian Bayes-factor design analysis.

**Simulate data**

The meta-analysis by Epp et al. (2012) found that clinically depressed participants responded more slowly than control participants to negative Stroop stimuli. We use their reported effect size to showcase how data for two groups can be simulated.

The function combines Hedges' g with an illustrative control-group mean and pooled standard deviation. The two group labels can be changed as needed. We save the simulated data in `data/` so they can also be inspected outside this notebook.

In [ ]:
N = 100
control_label, experimental_label = "control", "depressed"
df = simulate_stroop(n=N, con_label=control_label, exp_label=experimental_label)
df.to_csv("data/simulated_stroop.csv", index=False)
df.head()

**Power analysis**

Frequentist power is the probability of obtaining a significant result when the assumed effect exists. The Bayesian design analysis instead gives the probability of reaching a chosen BF10 evidence threshold.

In [ ]:
power, power_n = stroop_power(n=N, g=HEDGES_G, desired_power=0.8, alpha=0.05)
bf_probability, bf_n = stroop_bayes_design(n=N, g=HEDGES_G, desired_probability=0.8, target_bf=10)

print(f"Frequentist: power = {power:.2f}, required n = {power_n} per group")
print(f"Bayesian: P(BF10 ≥ 10) = {bf_probability:.2f}, required n = {bf_n} per group")

For convenience, our Dash app combines the simulation and both design analyses. The controls show how assumptions about the sample, alpha, power, and Bayes-factor threshold affect the required sample size.

In [ ]:
app.run(debug=False, port=8050)

## 3. Run your study

We can run the Stroop experiment from this notebook. Each participant's trial-level data are saved as a separate CSV file in `data/`.

In [ ]:
participant_id = "p-002"
n_trials = 20
run_stroop(participant_id=participant_id, n_trials=n_trials)

## 4. Analyse data

The simulated and collected data answer different questions, so we analyse them separately. Both routes include a visualization and frequentist and Bayesian t-tests.

**Simulated group data**

We first inspect the group distributions. We then compare the two simulated groups using Welch's t-test and a two-sided JZS Bayesian t-test.

In [ ]:
stroop_histplot(df)
frequentist_test = stroop_t_test(df, con_label=control_label, exp_label=experimental_label)
bayesian_test = stroop_bayes_t_test(df, con_label=control_label, exp_label=experimental_label)

**Collected experiment data**

We combine all participant files from `data/`. For correct trials, the analysis compares each participant's mean reaction time on incongruent and congruent trials using paired frequentist and Bayesian t-tests.

In [ ]:
experiment_df = load_experiment_data()
experiment_df.head()

In [ ]:
if experiment_df["participant_id"].nunique() < 2:
    print("Run at least two participants before analysing the experiment.")
else:
    stroop_experiment_plot(experiment_df)
    experiment_test = stroop_experiment_t_test(experiment_df)